## Install Dependencies
Install moviepy:

In [ ]:
!pip install moviepy

## Create Project Files

In [ ]:
import os
import json

# Create project folders
os.makedirs("storyboard_generator/outputs", exist_ok=True)

# prompts.json
prompts = [
    "A cyberpunk city at night with glowing neon signs",
    "A serene forest with a waterfall and mist",
    "A futuristic control room inside a spaceship",
    "A medieval castle during sunset with flying dragons",
    "A close-up shot of a robot painting on canvas"
]
with open("storyboard_generator/prompts.json", "w") as f:
    json.dump(prompts, f, indent=4)

# generate_images_with_comfyui.py
with open("storyboard_generator/generate_images_with_comfyui.py", "w") as f:
    f.write("""
import os
import json

def generate_images(prompts, output_dir):
    for i, prompt in enumerate(prompts):
        with open(os.path.join(output_dir, f"scene_{i+1}.txt"), "w") as f:
            f.write(f"Generated image for prompt: {prompt}")

if __name__ == "__main__":
    os.makedirs("outputs", exist_ok=True)
    with open("prompts.json", "r") as f:
        prompts = json.load(f)
    generate_images(prompts, "outputs")
""")

# assemble_video.py
with open("storyboard_generator/assemble_video.py", "w") as f:
    f.write("""
from moviepy.editor import ImageSequenceClip
import os

def create_video_from_images(image_folder, output_file, fps=1):
    images = sorted([os.path.join(image_folder, img) for img in os.listdir(image_folder) if img.endswith(".png")])
    if not images:
        print("No images found. Add generated PNGs to the output folder.")
        return
    clip = ImageSequenceClip(images, fps=fps)
    clip.write_videofile(output_file)

if __name__ == "__main__":
    create_video_from_images("outputs", "storyboard_video.mp4")
""")


In [ ]:
!pip install diffusers transformers accelerate safetensors

In [ ]:
from huggingface_hub import login
login("HF")

In [ ]:
from diffusers import StableDiffusionPipeline
import torch
from PIL import Image

# Load prompts
with open("storyboard_generator/prompts.json", "r") as f:
    prompts = json.load(f)

# Prepare output directory
output_dir = "storyboard_generator/outputs"
os.makedirs(output_dir, exist_ok=True)

# Load the Stable Diffusion pipeline
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",  # Or another SD model
    torch_dtype=torch.float16
).to("cuda" if torch.cuda.is_available() else "cpu")

# Generate images
for i, prompt in enumerate(prompts):
    print(f"Generating image {i+1}/{len(prompts)}: {prompt}")
    image = pipe(prompt).images[0]
    image.save(os.path.join(output_dir, f"scene_{i+1}.png"))

print("All images generated and saved to 'outputs/' folder.")

## Run Video Assembly (with actual PNGs)

In [ ]:
from moviepy.editor import ImageSequenceClip
import os

def create_video_from_images(image_folder, output_file, fps=1):
    images = sorted([os.path.join(image_folder, img) for img in os.listdir(image_folder) if img.endswith(".png")])
    if not images:
        print("No images found. Add generated PNGs to the output folder.")
    else:
        clip = ImageSequenceClip(images, fps=fps)
        clip.write_videofile(output_file)

create_video_from_images("storyboard_generator/outputs", "storyboard_video.mp4")

## Save the Final Video

In [ ]:
from google.colab import files
files.download("storyboard_video.mp4")